# Restitution en 3 actes — scaffold déterministe, narration LLM *gated*

> Série **Argument Analysis** · distillation de l'arc *anti-théâtre / fail-loud* de
> [`2025-Epita-Intelligence-Symbolique`](https://github.com/jsboige/2025-Epita-Intelligence-Symbolique)
> (Epic source #1258 / #1134). Voir l'Epic CoursIA **#2137**.

Une analyse argumentative produit un **dump dimensionnel** : des arguments, des sophismes,
des scores de qualité, des verdicts de solveur, des extensions de Dung. Illisible pour un
non-spécialiste. La **restitution** transforme ce dump en un **récit en trois actes** qu'un
lecteur peut suivre :

| Acte | Rôle |
|------|------|
| **Acte I — Mise en situation** | le cadre : genre du discours, enjeux, spectre de sophismes attendu (le seul acte qui *anticipe*) |
| **Acte II — Récit dialectique** | le cœur : la narration découpée par mouvement argumentatif, tissant qualité + sophismes + contre-arguments + tenue formelle |
| **Acte III — Conclusion actionnable** | le verdict *gated*, les appréciations (forces **et** faiblesses), et « comment se faire son avis » |

Mais cette narration a besoin d'un **LLM** — et un LLM laissé seul **fabrique**, **sur-affirme**,
**fait fuiter du jargon**. La question centrale de ce notebook :

> **Comment obtenir un rapport *lisible* dont l'*honnêteté est garantie*, que le LLM soit
> disponible ou non ?**

La réponse est une **séparation des responsabilités** :

- un **scaffold 100% déterministe** détient le *contrat d'honnêteté* (l'extraction ne surface que du
  réel-en-état, la bande de verdict est *gated* sur la couverture réelle, le renderer **nomme** les actes
  manquants, le gate de lisibilité interdit mécaniquement l'énumération) ;
- le **LLM** détient *uniquement* la **prose**, et il est **gated** : quand il est absent, le rapport le
  **dit bruyamment** (fail-loud) — **jamais** un template (anti-pendule #1108/#1019).

Tout ce notebook est **pur stdlib Python** (aucun LLM, aucune JVM, aucun solveur n'est requis pour
l'exécuter). Le scaffold déterministe **tourne entièrement ici** ; le LLM est démontré par son **contrat
injectable** et son **comportement fail-loud**.


## 1. Le contrat des 3 actes

`RestitutionActs` est un conteneur volontairement minuscule et sans dépendance : il **découple** le
renderer du *comment* chaque acte est généré. Deux invariants d'honnêteté y sont câblés dès le contrat :

- un acte **vide** est traité comme *manquant* — le renderer le **nomme** (« acte indisponible »), il ne
  l'omet **jamais** silencieusement ;
- `source_id` est **opaque** (jamais un nom de locuteur, un titre ou une date) — la confidentialité est
  *HARD*, elle commence dès le type de données.


In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Awaitable, Callable, Dict, List, Optional
import re

ACT_TITLES: Dict[int, str] = {
    1: "Acte I — Mise en situation",
    2: "Acte II — Récit dialectique",
    3: "Acte III — Conclusion actionnable",
}

@dataclass
class RestitutionActs:
    """Les trois actes narratifs + l'identifiant OPAQUE du corpus."""
    act1_framing: str = ""
    act2_narrative: str = ""
    act3_conclusion: str = ""
    source_id: str = ""               # opaque : jamais nom/titre/date (vie privée HARD)
    degraded: Dict[str, str] = field(default_factory=dict)

    def as_dict(self) -> Dict[int, str]:
        return {1: self.act1_framing, 2: self.act2_narrative, 3: self.act3_conclusion}

    @staticmethod
    def act_key(n: int) -> str:
        return {1: "act1_framing", 2: "act2_narrative", 3: "act3_conclusion"}[n]

    def is_missing(self, n: int) -> bool:
        """Vrai si l'acte n est absent (vide / blancs) -> le renderer le NOMME."""
        return not (self.as_dict()[n] or "").strip()

print("Contrat des 3 actes :")
for n, t in ACT_TITLES.items():
    print(f"  {n}. {t}")


Contrat des 3 actes :
  1. Acte I — Mise en situation
  2. Acte II — Récit dialectique
  3. Acte III — Conclusion actionnable


## 2. La bande de verdict — *gated* sur la couverture réelle

Un rapport ne doit jamais affirmer **plus fort** que ce que l'analyse a réellement couvert. La **bande de
verdict** gouverne *la force avec laquelle le discours peut être caractérisé*, dérivée du **nombre d'axes
analytiques non-triviaux** réellement présents (sophismes, qualité, contre-arguments, formel PL, formel
FOL, Dung).

Les seuils sont **transparents et fixes** (anti-pendule : pas de courbe ajustée *a posteriori*). `EXCEEDED`
exige en plus la **profondeur formelle** (PL ou FOL vérifié) **ET** l'axe **qualité** — une couverture large
sans profondeur ne « dépasse » rien. Adapté de #1008 §2.1 : la restitution n'a pas d'analyste externe à qui
se comparer, donc la bande mesure la couverture *interne*.


In [2]:
AXES = ["fallacies", "quality", "counter_arguments", "formal_pl", "formal_fol", "dung"]

@dataclass
class VerdictBand:
    band: str
    nontrivial_axes: List[str] = field(default_factory=list)
    missing_axes: List[str] = field(default_factory=list)
    axes_count: int = 0

def compute_verdict_band(axes_nontrivial: List[str]) -> VerdictBand:
    present = set(axes_nontrivial)
    missing = [a for a in AXES if a not in present]
    n = len(axes_nontrivial)
    has_formal = "formal_pl" in present or "formal_fol" in present
    has_quality = "quality" in present
    if n >= 5 and has_formal and has_quality:
        band = "EXCEEDED"
    elif n >= 4:
        band = "MATCH"
    elif n >= 2:
        band = "PARTIAL"
    else:
        band = "BELOW"
    return VerdictBand(band, list(axes_nontrivial), missing, n)

# Démo : la même mécanique sur 4 niveaux de couverture
for axes in (
    ["fallacies", "quality", "counter_arguments", "formal_pl", "formal_fol", "dung"],
    ["fallacies", "counter_arguments", "formal_pl", "formal_fol", "dung"],   # 5 mais SANS qualité
    ["fallacies", "quality", "dung"],
    ["fallacies"],
):
    v = compute_verdict_band(axes)
    print(f"  {v.axes_count}/6 axes -> {v.band:9s}  (touchés: {', '.join(v.nontrivial_axes)})")


  6/6 axes -> EXCEEDED   (touchés: fallacies, quality, counter_arguments, formal_pl, formal_fol, dung)
  5/6 axes -> MATCH      (touchés: fallacies, counter_arguments, formal_pl, formal_fol, dung)
  3/6 axes -> PARTIAL    (touchés: fallacies, quality, dung)
  1/6 axes -> BELOW      (touchés: fallacies)


Noter le 2ᵉ cas : **5 axes mais sans l'axe qualité → `MATCH`, pas `EXCEEDED`**. La règle refuse de
« dépasser » sans un discours caractérisé. C'est le contraire d'un score qui monte mécaniquement avec le
volume.

## 3. L'extraction d'evidence — déterministe, *réel-en-état seulement*

`build_evidence(state)` lit un **état d'analyse partagé** (la sortie des notebooks de détection / formels /
Dung de cette série) et en extrait un **paquet d'evidence** : points faibles localisés, stratégies de
contre, forces de qualité, extraits de revendications. Trois disciplines :

- **réel-en-état seulement** (G4) : aucun axe n'est fabriqué ; un verdict formel non vérifié reste `None`,
  jamais collapsé en `False` (`None ≠ False`, #1019) ;
- chaque **point faible** est lié à *(a)* un argument localisé **ET** *(b)* le verdict concret qui l'a
  signalé — c'est l'ancrage qu'exigera le gate de lisibilité ;
- les champs dérivés du corpus sont **tronqués** (vie privée + budget de prompt) et les IDs restent
  **opaques**.


In [3]:
@dataclass
class WeakPoint:
    source: str            # "fallacy" | "pl" | "fol" | "dung"
    label: str
    target_arg_id: str
    detail: str = ""

@dataclass
class Evidence:
    args_total: int = 0
    fallacies_total: int = 0
    counters_total: int = 0
    quality_axis_available: bool = False
    quality_strengths: List[tuple] = field(default_factory=list)     # (vertu, score)
    weak_points: List[WeakPoint] = field(default_factory=list)
    counter_strategies: List[tuple] = field(default_factory=list)    # (cible, stratégie, extrait)
    claim_excerpts: List[str] = field(default_factory=list)
    verdict: Optional[VerdictBand] = None
    gates: Dict[str, bool] = field(default_factory=dict)

_CLAIM_CAP, _DETAIL_CAP = 240, 200

def _truncate(text: Any, cap: int) -> str:
    if not text:
        return ""
    s = str(text).strip()
    return s if len(s) <= cap else s[:cap].rstrip() + " […]"

def _pl_verdict(r: Dict[str, Any]) -> Optional[bool]:
    """Verdict PL en bool STRICT (None si non vérifié) — jamais bool()."""
    sat = r.get("satisfiable")
    if sat is None:
        sat = r.get("consistent")
    return sat if isinstance(sat, bool) else None

def _dung_rejected(state: Dict[str, Any]) -> Dict[str, str]:
    """arg_id -> sémantique, pour les arguments présents mais hors extension."""
    rejected: Dict[str, str] = {}
    for _fid, fw in (state.get("dung_frameworks") or {}).items():
        if not isinstance(fw, dict):
            continue
        accepted = set(fw.get("extension", []) or [])
        sem = str(fw.get("semantics", "grounded"))
        for a in (fw.get("arguments", []) or []):
            if a not in accepted:
                rejected.setdefault(a, sem)
    return rejected

def build_evidence(state: Dict[str, Any]) -> Evidence:
    args = state.get("identified_arguments") or {}
    fallacies = state.get("identified_fallacies") or {}
    quality = state.get("argument_quality_scores") or {}
    counters = state.get("counter_arguments") or []
    pl = state.get("propositional_analysis_results") or []
    fol = state.get("fol_analysis_results") or []

    fallacies_total = sum(1 for d in fallacies.values()
                          if isinstance(d, dict) and d.get("target_argument_id"))
    counters_total = sum(1 for c in counters
                         if isinstance(c, dict) and (c.get("target_arg_id") or c.get("counter_content")))
    pl_inc = sum(1 for r in pl if isinstance(r, dict) and _pl_verdict(r) is False)
    fol_inc = sum(1 for r in fol if isinstance(r, dict) and r.get("consistent") is False)
    pl_verified = sum(1 for r in pl if isinstance(r, dict) and _pl_verdict(r) is not None)
    fol_verified = sum(1 for r in fol if isinstance(r, dict) and r.get("consistent") in (True, False))
    dung_rejected = _dung_rejected(state)

    axes: List[str] = []
    if fallacies_total:            axes.append("fallacies")
    if bool(quality):              axes.append("quality")
    if counters_total:             axes.append("counter_arguments")
    if pl_verified:                axes.append("formal_pl")
    if fol_verified:               axes.append("formal_fol")
    if dung_rejected:              axes.append("dung")

    weak: List[WeakPoint] = []
    for _fid, fd in fallacies.items():
        if isinstance(fd, dict) and fd.get("target_argument_id"):
            weak.append(WeakPoint("fallacy",
                f"{fd.get('type','inconnu')} (famille {fd.get('family','inconnu')})",
                str(fd["target_argument_id"]), _truncate(fd.get("justification", ""), _DETAIL_CAP)))
    for aid, sem in dung_rejected.items():
        weak.append(WeakPoint("dung", f"argument rejeté par le cadre de Dung (sémantique {sem})", str(aid)))
    if pl_inc:
        weak.append(WeakPoint("pl", f"{pl_inc} inférence(s) propositionnelle(s) inconsistantes (solveur Tweety)", "—"))
    if fol_inc:
        weak.append(WeakPoint("fol", f"{fol_inc} théorie(s) du premier ordre inconsistantes (solveur Tweety)", "—"))

    best: Dict[str, float] = {}
    for qs in quality.values():
        spv = qs.get("scores") if isinstance(qs, dict) and isinstance(qs.get("scores"), dict) else {}
        for vname, vval in spv.items():
            if isinstance(vval, (int, float)) and vval > 0:
                best[vname] = max(best.get(vname, 0.0), float(vval))
    strengths = sorted(best.items(), key=lambda kv: kv[1], reverse=True)

    counter_strategies = [
        (str(c.get("target_arg_id", "")), str(c.get("strategy", "")), _truncate(c.get("counter_content", ""), _DETAIL_CAP))
        for c in counters if isinstance(c, dict) and (c.get("target_arg_id") or c.get("counter_content"))
    ]
    claim_excerpts = [_truncate(v, _CLAIM_CAP) for v in list(args.values())[:5] if _truncate(v, _CLAIM_CAP)]

    gates = {
        "G1_arguments_extracted": len(args) > 0,
        "G2_one_dimension_nontrivial": len(axes) >= 1,
        "G3_verdict_computed": True,
        "G4_no_fabrication": True,
    }
    return Evidence(len(args), fallacies_total, counters_total, bool(quality),
                    strengths, weak, counter_strategies, claim_excerpts,
                    compute_verdict_band(axes), gates)


### Un état d'analyse de travail

On se donne un état **réaliste** : trois arguments (un appel à l'autorité, un appel à la popularité, une
revendication chiffrée), deux sophismes localisés, des scores de qualité sur l'argument chiffré, un
contre-argument, un résultat PL cohérent, une théorie FOL **in**cohérente, et un cadre de Dung qui rejette
deux arguments. C'est exactement la forme produite par les notebooks `1-informal` / `2-formal` /
`3-orchestration` de cette série.

In [4]:
ETAT = {
    "identified_arguments": {
        "arg_1": "Les experts du comité affirment que la réforme est sûre, donc elle l'est.",
        "arg_2": "Tout le monde adopte cette mesure ; vous devriez l'adopter aussi.",
        "arg_3": "La transition réduit les coûts de 12% sur trois ans selon le rapport budgétaire.",
    },
    "identified_fallacies": {
        "f_1": {"type": "appel à l'autorité", "family": "ad verecundiam", "target_argument_id": "arg_1",
                "justification": "L'autorité invoquée n'est pas compétente sur ce point précis."},
        "f_2": {"type": "appel à la popularité", "family": "ad populum", "target_argument_id": "arg_2",
                "justification": "L'adoption massive ne fonde pas la validité de la mesure."},
    },
    "argument_quality_scores": {
        "arg_3": {"scores": {"clarté": 0.8, "pertinence": 0.7, "rigueur": 0.6}},
    },
    "counter_arguments": [
        {"target_arg_id": "arg_1", "strategy": "réfutation directe",
         "counter_content": "Le comité cité a un conflit d'intérêts documenté sur ce dossier."},
    ],
    "propositional_analysis_results": [{"satisfiable": True}],
    "fol_analysis_results": [{"consistent": False}],
    "dung_frameworks": {"af_1": {"arguments": ["arg_1", "arg_2", "arg_3"],
                                 "extension": ["arg_3"], "semantics": "grounded"}},
}

ev = build_evidence(ETAT)
print(f"Bande de verdict : {ev.verdict.band}  ({ev.verdict.axes_count}/6 axes)")
print(f"Axes touchés    : {', '.join(ev.verdict.nontrivial_axes)}")
print(f"Gates G1-G4     : {ev.gates}")
print("\nPoints faibles localisés :")
for wp in ev.weak_points:
    print(f"  - [{wp.source}] {wp.label} -> {wp.target_arg_id}")
print("\nForces (qualité) :", ", ".join(f"{v} {s:.1f}" for v, s in ev.quality_strengths))


Bande de verdict : EXCEEDED  (6/6 axes)
Axes touchés    : fallacies, quality, counter_arguments, formal_pl, formal_fol, dung
Gates G1-G4     : {'G1_arguments_extracted': True, 'G2_one_dimension_nontrivial': True, 'G3_verdict_computed': True, 'G4_no_fabrication': True}

Points faibles localisés :
  - [fallacy] appel à l'autorité (famille ad verecundiam) -> arg_1
  - [fallacy] appel à la popularité (famille ad populum) -> arg_2
  - [dung] argument rejeté par le cadre de Dung (sémantique grounded) -> arg_1
  - [dung] argument rejeté par le cadre de Dung (sémantique grounded) -> arg_2
  - [fol] 1 théorie(s) du premier ordre inconsistantes (solveur Tweety) -> —

Forces (qualité) : clarté 0.8, pertinence 0.7, rigueur 0.6


## 4. Le gate de lisibilité — la **règle de tissage** (§4) en code

C'est le **contrat anti-énumération** du rapport. La spec §4 dit : *pour chaque citation d'un cadre (Tweety,
Dung, taxonomie, vertus), le rapport DOIT fournir un ancrage narratif — le cadre est la preuve d'un point de
récit, jamais une entrée de liste isolée.*

Le détecteur repère l'**odeur d'énumération** — le contre-exemple canonique :

> ❌ « Sophisme : ad verecundiam (0.8) » — un nom + un nombre, détaché de toute histoire.

Une ligne est **nue** quand elle cite un cadre **ET** porte un score isolé **ET** ne contient aucun *verbe
narratif*. Le contraste :

> ✅ « … le solveur Tweety **confirme** l'inconsistance de l'inférence. » — cite Tweety, porte un verbe, pas
> de score isolé → *tissée*.

Le gate **reporte** ce qu'il trouve (WARN pour un résidu, FAIL pour une énumération manifeste) ; il ne
**truque jamais** un PASS pour plaire (#1019).


In [5]:
_FRAMEWORKS = {"tweety", "dung", "aspic", "aif", "walton", "jtms", "atms"}
_FALLACY_NAMES = ("ad hominem", "ad verecundiam", "ad populum", "ad baculum", "tu quoque",
                  "post hoc", "petitio", "non sequitur", "slippery slope", "straw man",
                  "false dichotomy", "faux dilemme", "sophisme", "paralogisme")
_NARRATIVE_VERBS = ("confirme", "invalide", "valide", "montre", "prouve", "démontre", "isole",
                    "défait", "appuie", "soutient", "implique", "révèle", "indique", "établit",
                    "signifie", "traduit", "reflète", "expose", "illustre", "n'est", "c'est",
                    "permet", "justifie", "constitue")
_STANDALONE_SCORE_RE = re.compile(
    r"\(\s*0?\.\d{1,2}\s*\)"
    r"|\(?\s*(?:score|confiance|poids|confidence|sévérité)\s*[:=]?\s*0?\.\d{1,2}\b")
_DUMP_HEADING_RE = re.compile(
    r"^\s*#{0,6}\s*(?:Sophisme|Argument|Dimension|Phase)\s+\d+\s*[:\-—]?",
    re.IGNORECASE | re.MULTILINE)

@dataclass
class GateVerdict:
    band: str  # PASS | WARN | FAIL
    reasons: List[str] = field(default_factory=list)
    @property
    def passed(self) -> bool:
        return self.band in ("PASS", "WARN")
    def merge(self, other: "GateVerdict") -> "GateVerdict":
        order = {"PASS": 0, "WARN": 1, "FAIL": 2}
        worst = self if order[self.band] >= order[other.band] else other
        return GateVerdict(worst.band, [*self.reasons, *other.reasons])

def _worsen(a: str, b: str) -> str:
    order = {"PASS": 0, "WARN": 1, "FAIL": 2}
    return a if order[a] >= order[b] else b

def _line_has_framework(line: str) -> bool:
    low = line.lower()
    return any(k in low for k in _FRAMEWORKS) or any(n in low for n in _FALLACY_NAMES)

def _line_is_bare(line: str) -> bool:
    """Cadre cité + score isolé + AUCUN verbe narratif = référence NUE."""
    if not _line_has_framework(line):
        return False
    if not _STANDALONE_SCORE_RE.search(line):
        return False
    return not any(v in line.lower() for v in _NARRATIVE_VERBS)

class ReadabilityGate:
    def __init__(self, bare_warn=2, bare_fail=3, dump_fail=2):
        self.bare_warn, self.bare_fail, self.dump_fail = bare_warn, bare_fail, dump_fail
    def check_acts(self, acts: RestitutionActs) -> GateVerdict:
        reasons, worst, total_bare = [], "PASS", 0
        for n in (1, 2, 3):
            text = acts.as_dict()[n] or ""
            title = {1: "Acte I", 2: "Acte II", 3: "Acte III"}[n]
            if acts.is_missing(n):
                reasons.append(f"{title} absent — le rapport n'est pas complet (générateur non câblé ou acte vide).")
                worst = _worsen(worst, "FAIL"); continue
            bare = [ln.strip() for ln in text.splitlines() if _line_is_bare(ln)]
            total_bare += len(bare)
            if bare:
                reasons.append(f"{title}: {len(bare)} référence(s) de cadre « nue(s) » "
                               f"(framework + score isolé, sans ancrage narratif — §4). Ex: « {bare[0][:90]} ».")
            dump_n = len(_DUMP_HEADING_RE.findall(text))
            if dump_n >= self.dump_fail:
                reasons.append(f"{title}: {dump_n} titres « Sophisme N: / Argument N: » — l'acte a dégénéré en énumération.")
                worst = _worsen(worst, "FAIL")
        if total_bare >= self.bare_fail:   worst = _worsen(worst, "FAIL")
        elif total_bare > 0:               worst = _worsen(worst, "WARN")
        return GateVerdict(worst, reasons)

gate = ReadabilityGate()
woven = RestitutionActs(
    act1_framing="Le discours se présente comme un plaidoyer technique au ton assuré. " * 3,
    act2_narrative="Le solveur Tweety confirme l'inconsistance de cette inférence, ce qui défait l'argument central du plaidoyer. " * 2,
    act3_conclusion="En conclusion, le cadre de Dung isole cette revendication comme rejetée ; le lecteur doit la recevoir avec prudence. " * 2)
bare = RestitutionActs(
    act1_framing="Mise en situation correcte et suffisamment développée du discours technique. " * 3,
    act2_narrative="ad verecundiam (0.8)\nad populum (0.7)\ndung (0.9)",   # nues : nom + score, aucun verbe
    act3_conclusion="Conclusion actionnable détaillée à destination du lecteur, nuancée. " * 3)
print("Actes TISSÉS :", gate.check_acts(woven).band)
vb = gate.check_acts(bare)
print("Actes NUS    :", vb.band)
for r in vb.reasons:
    print("   -", r)


Actes TISSÉS : PASS
Actes NUS    : FAIL
   - Acte II: 3 référence(s) de cadre « nue(s) » (framework + score isolé, sans ancrage narratif — §4). Ex: « ad verecundiam (0.8) ».


## 5. Le renderer *fail-loud*

Le renderer assemble les trois actes en un Markdown lisible — et c'est ici que l'honnêteté devient visible.
Un acte **manquant** est **nommé** (« Acte II indisponible — … »), jamais omis. Un acte **dégradé** est émis
**avec sa note** de dégradation. Et le **verdict du gate** est surfacé *verbatim* dans le rapport : le
document ne se déclare pas lisible si le gate n'est pas d'accord.

Démontrons-le sur le cas le plus instructif : un rapport où **seul l'Acte I** a été produit (Actes II et III
non câblés). Au lieu d'un rapport tronqué qui *semble* complet, on obtient un rapport qui **dit** ce qui
manque.


In [6]:
_MIN_ACT_CHARS = 120
_MISSING_WORDING = {
    1: ("Acte I indisponible — le générateur de mise en situation n'a pas produit de cadre "
        "(non câblé ou en échec). Le rapport entre directement dans l'analyse."),
    2: ("Acte II indisponible — le récit dialectique n'a pas pu être généré "
        "(cœur narratif absent). Le rapport n'a pas de substance narrative."),
    3: ("Acte III indisponible — la conclusion actionnable n'a pas été générée "
        "(portes G1–G4 non évaluées). Le rapport s'arrête sans synthèse."),
}

@dataclass
class RenderedReport:
    markdown: str
    verdict: GateVerdict

def _render_verdict_block(v: GateVerdict) -> str:
    icon = {"PASS": "[PASS]", "WARN": "[WARN]", "FAIL": "[FAIL]"}.get(v.band, "[?]")
    lines = ["---", "", f"## Gate lisibilité — {icon} {v.band}", ""]
    if v.reasons:
        lines += ["Contrôles structurels (règle de tissage, §4) :", ""]
        lines += [f"- {r}" for r in v.reasons]
    else:
        lines.append("Tous les contrôles structurels passent : 3 actes présents et non-triviaux, aucune référence de cadre nue.")
    lines += ["", "_Verdict honnête reporté par le gate — non truqué (#1019)._"]
    return "\n".join(lines)

def render_report(acts: RestitutionActs, gate: Optional[ReadabilityGate] = None) -> RenderedReport:
    gate = gate or ReadabilityGate()
    body, thin = [], []
    for n in (1, 2, 3):
        body += [f"## {ACT_TITLES[n]}", ""]
        if acts.is_missing(n):
            body += [f"_{_MISSING_WORDING[n]}_", ""]            # fail-loud : NOMME l'acte manquant
            continue
        text = acts.as_dict()[n].strip()
        body += [text, ""]
        key = RestitutionActs.act_key(n)
        if key in acts.degraded:
            body += [f"> [!WARNING] **Acte dégradé** — {acts.degraded[key]}", ""]
        if len(text) < _MIN_ACT_CHARS:
            thin.append(f"{ACT_TITLES[n]} est anormalement court ({len(text)} caractères) — sortie probablement stub/dégradée.")
    verdict = gate.check_acts(acts)
    if thin:
        verdict = verdict.merge(GateVerdict("WARN", thin))
    source = (acts.source_id or "corpus_anonyme").strip()
    parts = [
        f"# Rapport de restitution — {source}", "",
        ("Récit en trois actes (mise en situation -> analyse narrative -> conclusion actionnable). "
         "Les cadres formels/informels (Tweety, Dung, taxonomie, vertus) sont les *preuves* citées en "
         "appui du récit, jamais une énumération (§4)."), "",
        "\n".join(body).strip(), "",
        _render_verdict_block(verdict), "",
    ]
    return RenderedReport("\n".join(parts).rstrip() + "\n", verdict)

partiel = RestitutionActs(
    act1_framing=("Le discours relève du plaidoyer institutionnel : il défend une réforme en s'appuyant "
                  "sur l'autorité d'un comité et sur l'adhésion générale. On peut s'attendre à des appuis "
                  "d'autorité et de popularité, classiques de ce registre."),
    source_id="doc_A")   # Actes II et III NON câblés
rendu = render_report(partiel)
print(rendu.markdown)


# Rapport de restitution — doc_A

Récit en trois actes (mise en situation -> analyse narrative -> conclusion actionnable). Les cadres formels/informels (Tweety, Dung, taxonomie, vertus) sont les *preuves* citées en appui du récit, jamais une énumération (§4).

## Acte I — Mise en situation

Le discours relève du plaidoyer institutionnel : il défend une réforme en s'appuyant sur l'autorité d'un comité et sur l'adhésion générale. On peut s'attendre à des appuis d'autorité et de popularité, classiques de ce registre.

## Acte II — Récit dialectique

_Acte II indisponible — le récit dialectique n'a pas pu être généré (cœur narratif absent). Le rapport n'a pas de substance narrative._

## Acte III — Conclusion actionnable

_Acte III indisponible — la conclusion actionnable n'a pas été générée (portes G1–G4 non évaluées). Le rapport s'arrête sans synthèse._

---

## Gate lisibilité — [FAIL] FAIL

Contrôles structurels (règle de tissage, §4) :

- Acte II absent — le rapport n'est pas complet 

Le rapport est **complet dans sa structure** tout en étant **honnête sur ses trous** : un lecteur sait
exactement ce qui a été produit et ce qui manque. Le verdict `[FAIL]` du gate n'est pas un bug — c'est le
rapport qui **refuse de se déclarer lisible** alors que deux actes manquent.

## 6. La frontière LLM — prompt *conduit*, callable injectable, *fail-loud*

Jusqu'ici, **tout est déterministe** et a tourné entièrement. La **narration** de chaque acte, elle, est
**conduite par un LLM**. La discipline (Epic source #1258) :

1. le **prompt** est construit *déterministiquement* à partir de l'evidence (`build_act3_prompt`) — il
   **varie avec le corpus**, ce n'est **pas** un template statique (#1108). On peut donc l'**inspecter** ;
2. le LLM est un **callable async injectable** `Callable[[str], Awaitable[str]]` (testable avec un stub,
   aucun kernel requis) ;
3. quand **aucun** LLM n'est injecté, l'orchestrateur est **fail-loud** : `status="unavailable"`, narration
   vide, et le renderer **nomme le trou**. **Jamais** de template de repli.

Affichons d'abord le **prompt conduit** que recevrait le LLM — entièrement dérivé de l'evidence réelle :


In [7]:
LlmCallable = Callable[[str], Awaitable[str]]
_BAND_CEILING = {
    "EXCEEDED": "Caractérise l'analyse comme APPROFONDIE et multi-axes (profondeur formelle vérifiée + profil de qualité).",
    "MATCH": "Caractérise l'analyse comme COMPLÈTE : elle couvre les axes principaux. Nomme honnêtement les axes non-concluables.",
    "PARTIAL": "Caractérise l'analyse comme PARTIELLE : diagnostic honnête, nomme ce qui est couvert ET ce qui manque.",
    "BELOW": "Caractérise l'analyse comme MINIMALE : couverture réduite, liste honnête des axes non-concluables.",
}

def build_act3_prompt(ev: Evidence) -> str:
    v = ev.verdict
    if v is not None:
        synth = (f"Bande de verdict : {v.band} (couverture : {v.axes_count}/6 axes non-triviaux).\n"
                 f"  Axes touchés : {', '.join(v.nontrivial_axes) or 'aucun'}.\n"
                 f"  Axes manquants : {', '.join(v.missing_axes) or 'aucun'}.\n"
                 f"  Plafond de claim autorisé : {_BAND_CEILING[v.band]}")
    else:
        synth = "Bande de verdict : NON CALCULABLE (gates G1–G4 non passés)."
    claims = "\n".join(f"  - {c}" for c in ev.claim_excerpts) or "  (aucune revendication extraite — G1 non passé)"
    strengths = "\n".join(f"  - {vir} (meilleur score : {sc:.1f})" for vir, sc in ev.quality_strengths[:6]) or "  (axe qualité non concluable)"
    weaknesses = "\n".join(f"  - [{wp.source}] {wp.label} sur {wp.target_arg_id}" + (f" : {wp.detail}" if wp.detail else "")
                           for wp in ev.weak_points[:8]) or "  (aucun point faible localisé — texte vertueux)"
    counters = "\n".join(f"  - Pour {t} ({s}) : {snip}" for t, s, snip in ev.counter_strategies[:8]) or "  (aucun contre-argument généré)"
    return (
        "Tu es l'auteur de l'ACTE III d'un rapport de restitution argumentative — la\n"
        "CONCLUSION ORIENTÉE LECTEUR. Trois battements en prose (titres thématiques en ###) :\n"
        "1. Ce que le discours dit vraiment (cite les revendications) ;\n"
        "2. Ce qui tient et ce qui ne tient pas (verdict en langage clair) ;\n"
        "3. Comment se faire son avis (points de prudence, pas des cibles à abattre).\n\n"
        "RÈGLE DE TISSAGE (§4) : chaque citation d'un cadre (Tweety, Dung, taxonomie, vertus)\n"
        "DOIT être ancrée narrativement — liée à un argument localisé ET au verdict concret.\n"
        "INTERDIT : « ad verecundiam (0.8) » (nom + score isolé). Le verdict formel APPUIE\n"
        "un battement, jamais une sous-section isolée.\n\n"
        "HONNÊTETÉ (anti-pendule #1019) : ne formule jamais un claim au-delà de la bande.\n"
        "Si un axe manque, dis-le ; ne le simule jamais. Aucun sophisme fabriqué pour remplir.\n\n"
        "DONNÉES VÉRIFIÉES DANS LE STATE (ne citer que celles-ci) :\n\n"
        f"[CE QUI A ÉTÉ DIT]\n{claims}\n\n"
        f"[VERDICT GATED]\n{synth}\n\n"
        f"[CE QUI TIENT — forces]\n{strengths}\n\n"
        f"[CE QUI NE TIENT PAS — faiblesses localisées]\n{weaknesses}\n\n"
        f"[CONTRE-POINTS]\n{counters}\n\n"
        "Rédige en français, 300-600 mots, markdown léger. La conclusion doit VARIER selon\n"
        "le contenu réel ci-dessus : pas de prose générique recyclable."
    )

print(build_act3_prompt(ev))


Tu es l'auteur de l'ACTE III d'un rapport de restitution argumentative — la
CONCLUSION ORIENTÉE LECTEUR. Trois battements en prose (titres thématiques en ###) :
1. Ce que le discours dit vraiment (cite les revendications) ;
2. Ce qui tient et ce qui ne tient pas (verdict en langage clair) ;
3. Comment se faire son avis (points de prudence, pas des cibles à abattre).

RÈGLE DE TISSAGE (§4) : chaque citation d'un cadre (Tweety, Dung, taxonomie, vertus)
DOIT être ancrée narrativement — liée à un argument localisé ET au verdict concret.
INTERDIT : « ad verecundiam (0.8) » (nom + score isolé). Le verdict formel APPUIE
un battement, jamais une sous-section isolée.

HONNÊTETÉ (anti-pendule #1019) : ne formule jamais un claim au-delà de la bande.
Si un axe manque, dis-le ; ne le simule jamais. Aucun sophisme fabriqué pour remplir.

DONNÉES VÉRIFIÉES DANS LE STATE (ne citer que celles-ci) :

[CE QUI A ÉTÉ DIT]
  - Les experts du comité affirment que la réforme est sûre, donc elle l'est.
  - Tou

Le prompt cite les **vraies revendications**, la **vraie bande**, les **vrais points faibles** — il est
**impossible** de le réutiliser tel quel sur un autre corpus. C'est le sens de « conduit, pas template ».

Maintenant l'**orchestrateur**, et son comportement quand **aucun LLM** n'est disponible (le cas par défaut
ici — aucune clé API n'est configurée) :


In [8]:
@dataclass
class Act3Result:
    narrative: str
    status: str   # woven | unavailable | empty_state
    degraded: Dict[str, str] = field(default_factory=dict)

async def build_act3_conclusion(state: Dict[str, Any], llm_callable: Optional[LlmCallable] = None) -> Act3Result:
    """L'evidence déterministe est TOUJOURS construite ; la conclusion est conduite par LLM
    SEULEMENT si llm_callable est fourni ; sinon fail-loud (jamais de template, #1108)."""
    ev = build_evidence(state)
    if not ev.gates["G1_arguments_extracted"]:
        return Act3Result("", "empty_state",
                          {"act3_conclusion": "Aucun argument extrait — pas de substrat à conclure (G1 échoué)."})
    if llm_callable is None:
        return Act3Result("", "unavailable",
                          {"act3_conclusion": "Conclusion non conduite — aucun LLM injecté pour l'Acte III (fail-loud, #1108)."})
    try:
        raw = await llm_callable(build_act3_prompt(ev))
    except Exception as exc:   # noqa: BLE001 — surface, don't fabricate
        return Act3Result("", "unavailable", {"act3_conclusion": f"Conclusion indisponible — le LLM a échoué : {exc} (fail-loud)."})
    if not raw:
        return Act3Result("", "unavailable", {"act3_conclusion": "Conclusion indisponible — le LLM n'a rien produit (fail-loud, #1108)."})
    return Act3Result(str(raw).strip(), "woven")

import asyncio, concurrent.futures

def run_async(coro):
    """Exécute une coroutine que l'on soit en script pur OU dans un kernel
    Jupyter (qui tourne déjà une boucle asyncio)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()

res = run_async(build_act3_conclusion(ETAT, llm_callable=None))   # aucun LLM -> fail-loud honnête
print("status   :", res.status)
print("narration:", repr(res.narrative))
print("dégradé  :", res.degraded["act3_conclusion"])

# On assène l'Acte III « indisponible » dans le rapport : il est NOMMÉ, jamais masqué.
acts_sans_llm = RestitutionActs(act1_framing=partiel.act1_framing, source_id="doc_A",
                                degraded=res.degraded)
print("\n--- rapport sans LLM (Acte III nommé comme indisponible) ---\n")
print(render_report(acts_sans_llm).markdown)


status   : unavailable
narration: ''
dégradé  : Conclusion non conduite — aucun LLM injecté pour l'Acte III (fail-loud, #1108).

--- rapport sans LLM (Acte III nommé comme indisponible) ---

# Rapport de restitution — doc_A

Récit en trois actes (mise en situation -> analyse narrative -> conclusion actionnable). Les cadres formels/informels (Tweety, Dung, taxonomie, vertus) sont les *preuves* citées en appui du récit, jamais une énumération (§4).

## Acte I — Mise en situation

Le discours relève du plaidoyer institutionnel : il défend une réforme en s'appuyant sur l'autorité d'un comité et sur l'adhésion générale. On peut s'attendre à des appuis d'autorité et de popularité, classiques de ce registre.

## Acte II — Récit dialectique

_Acte II indisponible — le récit dialectique n'a pas pu être généré (cœur narratif absent). Le rapport n'a pas de substance narrative._

## Acte III — Conclusion actionnable

_Acte III indisponible — la conclusion actionnable n'a pas été générée (portes G1

**C'est la sortie réelle et honnête sur une machine sans clé LLM** — exactement comme un prouveur externe
absent produit une sortie *fail-loud* plutôt qu'un verdict fabriqué (verdict SOTA **RECOVERABLE-USER-HAND** :
une clé API rebranche la narration réelle ; cf l'exercice 3). Le scaffold, lui, a produit *toute* l'evidence
vérifiable sans le moindre LLM.

### Visualiser un rapport *assemblé* — avec un conducteur illustratif

Pour **voir le renderer à l'œuvre sur un acte peuplé** sans clé LLM, on injecte ci-dessous un **conducteur
illustratif déterministe**. ⚠️ **En production c'est INTERDIT** (anti-pendule #1108) : seul un *vrai* LLM
fait varier la prose par corpus ; ce stand-in sert **uniquement** à illustrer l'assemblage, et le rapport le
**signale** via une note de dégradation. C'est la frontière même que la doctrine protège.


In [9]:
async def conducteur_illustratif(prompt: str) -> str:
    """STAND-IN PÉDAGOGIQUE — PAS un générateur de production. Produit une prose tissée
    FIXE pour montrer le renderer ; un vrai LLM varierait par corpus (#1108)."""
    return (
        "### Ce que le discours dit vraiment\n"
        "Le plaidoyer défend une réforme en s'appuyant sur l'autorité d'un comité et sur "
        "l'adhésion générale, tout en avançant une économie chiffrée de 12% sur trois ans.\n\n"
        "### Ce qui tient et ce qui ne tient pas\n"
        "L'argument chiffré est le plus solide ; mais l'analyse formelle invalide la chaîne de "
        "raisonnement central (théorie inconsistante), et le cadre de Dung isole deux revendications "
        "comme rejetées. L'appel à l'autorité ne tient pas : l'autorité invoquée n'est pas compétente.\n\n"
        "### Comment se faire son avis\n"
        "Le lecteur peut accueillir la donnée budgétaire avec confiance, mais doit recevoir avec "
        "prudence l'argument d'autorité et l'effet d'entraînement collectif — ce sont les points où le "
        "discours appuie sur l'adhésion plutôt que sur la preuve."
    )

res2 = run_async(build_act3_conclusion(ETAT, llm_callable=conducteur_illustratif))
print("status :", res2.status)

acts_complet = RestitutionActs(
    act1_framing=partiel.act1_framing,
    act2_narrative=("Le récit s'ouvre sur la donnée budgétaire, que rien ne contredit. Mais dès que le "
                    "plaidoyer invoque le comité, le solveur Tweety invalide l'inférence sous-jacente, et le "
                    "cadre de Dung défait l'argument d'autorité comme la revendication de popularité."),
    act3_conclusion=res2.narrative,
    source_id="doc_A",
    degraded={"act3_conclusion": "narration ILLUSTRATIVE (pédagogique) — en production, seul un vrai LLM remplit cet acte (#1108)."})
rendu_complet = render_report(acts_complet)
print("Verdict gate :", rendu_complet.verdict.band)
print("\n" + rendu_complet.markdown)


status : woven
Verdict gate : PASS

# Rapport de restitution — doc_A

Récit en trois actes (mise en situation -> analyse narrative -> conclusion actionnable). Les cadres formels/informels (Tweety, Dung, taxonomie, vertus) sont les *preuves* citées en appui du récit, jamais une énumération (§4).

## Acte I — Mise en situation

Le discours relève du plaidoyer institutionnel : il défend une réforme en s'appuyant sur l'autorité d'un comité et sur l'adhésion générale. On peut s'attendre à des appuis d'autorité et de popularité, classiques de ce registre.

## Acte II — Récit dialectique

Le récit s'ouvre sur la donnée budgétaire, que rien ne contredit. Mais dès que le plaidoyer invoque le comité, le solveur Tweety invalide l'inférence sous-jacente, et le cadre de Dung défait l'argument d'autorité comme la revendication de popularité.

## Acte III — Conclusion actionnable

### Ce que le discours dit vraiment
Le plaidoyer défend une réforme en s'appuyant sur l'autorité d'un comité et sur l'adh

Le rapport assemblé reste **honnête** : la note de dégradation signale explicitement que l'Acte III est
une narration *illustrative*. Le gate passe (`PASS`/`WARN`) parce que la prose est **tissée** (chaque cadre
porte un verbe, aucun score isolé) — exactement ce qu'un vrai LLM bien dirigé doit produire.

## 7. Exercices

Trois exercices pour étendre le scaffold. Chaque cellule est un **stub** à compléter — le notebook
s'exécute de bout en bout même si les exercices ne sont pas faits (la solution de référence n'est pas
fournie).

### Exercice 1 — Ajouter l'axe *logique modale* à la bande de verdict

Le state peut contenir une clé `modal_analysis_results` (liste de dicts `{"consistent": bool|None}`,
même forme que FOL). Étends la couverture pour qu'un résultat modal **vérifié** compte comme un 7ᵉ axe
non-trivial, et ajoute `"modal"` à la liste `AXES`. Veille à la discipline `None ≠ False` (un résultat
non vérifié ne compte pas).

In [10]:
def build_evidence_avec_modal(state):
    """Exercice 1 : recompter les axes en incluant un axe 'modal' vérifié.

    Indice : repars de build_evidence(state), puis inspecte
    state.get("modal_analysis_results") en comptant les entrées dont
    r.get("consistent") in (True, False)  # vérifié (None exclu).
    Recalcule compute_verdict_band(...) avec l'axe 'modal' ajouté si non-trivial.
    """
    # TODO étudiant : implémenter l'extension modale.
    print("Exercice 1 à completer : ajouter l'axe modal à la couverture.")
    return None


### Exercice 2 — Durcir le gate : interdire les *titres numérotés génériques*

Le détecteur actuel attrape « Sophisme N: » / « Argument N: ». Étends `_DUMP_HEADING_RE` (ou écris un
nouveau contrôle) pour **aussi** flaguer les titres génériques numérotés du type « ### Point 1 », « ###
Section 2 », « ### Étape 3 » qui trahissent un acte redevenu une liste plate. Renvoie un `GateVerdict`
`WARN` ou `FAIL` selon le nombre détecté.

In [11]:
def controle_titres_generiques(acte_markdown):
    r"""Exercice 2 : détecter les titres numérotés génériques (Point N / Section N / Étape N).

    Indice : une regex multiline ^#{1,6}\s*(?:Point|Section|Étape)\s+\d+ ; au-delà
    de 2 occurrences dans un acte -> GateVerdict('FAIL', [...]).
    """
    # TODO étudiant : implémenter le contrôle et renvoyer un GateVerdict.
    print("Exercice 2 à completer : flaguer les titres numérotés génériques.")
    return None


### Exercice 3 — Brancher un *vrai* LLM (gated sur ta clé)

Implémente un `llm_callable` réel et passe-le à `build_act3_conclusion(ETAT, llm_callable=...)`. C'est le
chemin **RECOVERABLE-USER-HAND** : il ne tourne que si **tu** fournis une clé API. **Ne hardcode jamais la
clé** — lis-la via `os.getenv("OPENAI_API_KEY")` (ou ton fournisseur), et si elle est absente, **laisse le
fail-loud** s'exprimer plutôt que de simuler une sortie.

In [12]:
import os

async def mon_llm(prompt):
    """Exercice 3 : conduire l'Acte III avec un vrai LLM (gated sur la clé).

    Indice : key = os.getenv("OPENAI_API_KEY"); si None -> raise / return None
    (laisse le fail-loud parler). Sinon, appeler l'API (openai / anthropic /
    endpoint local) avec `prompt` et renvoyer le texte de la complétion.
    NE JAMAIS hardcoder la clé ; NE JAMAIS renvoyer un template si la clé manque.
    """
    # TODO étudiant : appeler un vrai LLM et renvoyer sa complétion (str).
    print("Exercice 3 à completer : brancher un vrai LLM via une clé d'environnement.")
    return None


## Conclusion

Ce notebook a distillé le **contrat de restitution honnête** de l'arc *anti-théâtre* de
`2025-Epita-Intelligence-Symbolique` :

- un **scaffold déterministe** — extraction d'evidence (réel-en-état seulement), bande de verdict *gated*
  sur la couverture, gate de lisibilité (règle de tissage §4), renderer *fail-loud* — qui **tourne
  entièrement sans LLM** et **détient le contrat d'honnêteté** ;
- une **narration LLM gated** — prompt *conduit* (varie par corpus, pas un template), callable injectable,
  et **fail-loud** quand le LLM est absent (jamais un template de repli, #1108).

La leçon transférable : **la lisibilité d'un rapport peut être confiée à un LLM, mais son *honnêteté* doit
rester sous le contrôle d'un scaffold déterministe.** Le LLM enrichit la prose ; il ne décide jamais de ce
qui peut être affirmé, ni ne masque ce qui manque. Verdict SOTA : **SOTA-OK** pour le scaffold (pur Python
= le bon outil pour un contrat structurel) + **RECOVERABLE-USER-HAND** pour la narration (clé API exposée
honnêtement, jamais contournée).

**Pour aller plus loin** : voir `Argument_Analysis_Multi_Backend_Routing` (la sentinelle de contrat de
livraison côté *solveurs*), `Agentic-3-orchestration` (le pipeline qui peuple le state lu ici) et
`Agentic-5-jtms` (un autre moteur déterministe pur stdlib de cette série).
